In [5]:
# Standard Library
import random
import time

# Data Processing
import numpy as np
import pandas as pd
from numpy import inf

# Visualization
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import seaborn as sns
from mpl_toolkits import mplot3d

# Machine Learning
from sklearn import svm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import f1_score, accuracy_score
from sklearn.svm import SVC

# Dimensionality Reduction & Utilities
import umap
from joblib import Parallel, delayed

# Feature Selection
from testflows.combinatorics import Covering
from Py_FS.wrapper.population_based.BBA import BBA as BBA
from Py_FS.wrapper.population_based.PSO import PSO as PSO
from Py_FS.wrapper.population_based.CS import CS as CS

Load data set

In [17]:
df_cis = pd.read_csv('../data/cis_data.csv')
df_cacao = pd.read_csv(r'../data/data_temp/cacao.csv')
df_algarrobo = pd.read_csv(r'../data/data_temp/algarrobo.csv')
df_fruits_pures = pd.read_csv(r'../data/data_temp/MIR_Fruit_purees.csv')
df_fresh_meat = pd.read_csv(r'../data/data_temp/Fresh_meats.csv')
df_olive = pd.read_csv(r'../data/data_temp/Olive_Oils_Quadrum.csv')


df_x_cacao = df_cacao.iloc[:, 1:]
y_cacao = df_cacao.iloc[:, 0:1]
X_cacao = (df_x_cacao-df_x_cacao.min())/(df_x_cacao.max()-df_x_cacao.min())

unique_names_algarrobo = df_algarrobo['Labels'].unique()
algarrobo_x = df_algarrobo.loc[:, 'R':'REDVI']
y_algarrobo = df_algarrobo['Labels'].replace(to_replace=['N', 'P'], value=[0, 1]).to_frame()
X_algarrobo = (algarrobo_x-algarrobo_x.min())/(algarrobo_x.max()-algarrobo_x.min())

cis_x = df_cis[['X', 'Y', 'X10', 'Y10', 'X20', 'Y20', 'X30', 'Y30', 'X40', 'Y40']]
y_cis = df_cis[['Result']]

unique_names_berry = df_fruits_pures["label"].unique()
fruits_pures_x = df_fruits_pures.iloc[:,1:]
y_fruit_puree = df_fruits_pures.iloc[:,0:1].replace(to_replace=unique_names_berry, value =range(0,len(unique_names_berry)))
X_fruit_puree = (fruits_pures_x-fruits_pures_x.min())/(fruits_pures_x.max()-fruits_pures_x.min())


unique_names_meat = df_fresh_meat["meat"].unique()
meat_x = df_fresh_meat.iloc[:,4:]
y_meat = df_fresh_meat.iloc[:,0:1].replace(to_replace=unique_names_meat,value=range(0,len(unique_names_meat)))
X_meat =  (meat_x-meat_x.min())/(meat_x.max()-meat_x.min())

unique_names_olive = df_olive["Provenance"].unique()
olive_x = df_olive.iloc[:,3:]
y_olive = df_olive.iloc[:,2:3].replace(to_replace=unique_names_olive,value=range(0,len(unique_names_olive)))
X_olive =  (olive_x-olive_x.min())/(olive_x.max()-olive_x.min())

X_cis = (cis_x-cis_x.min())/(cis_x.max()-cis_x.min())
covering_array  = np.loadtxt(r'../data/coveringArray.csv', delimiter=",", dtype=int)

In [7]:
def ICAFS(dataset_X, dataset_Y, strenght,max_iteartion,model, print_logs=False):
  
  max_it = 1 

  v_variable = [0, 1]
  best_f1_score = float('-inf')
  max_iteartion_aux =  max_iteartion
  best_data_set = dataset_X.columns.values.copy()

  global_best_score = float('-inf')
  global_best_features = dataset_X.columns.values.copy()

  while max_iteartion_aux > 0:
      
      partial_score = 0
      dict_parameters = {}
      lst_of_featutres_to_check = []

      for colum_key in best_data_set:
          dict_parameters[colum_key] = v_variable
      generate_covering_array = Covering(dict_parameters, strength=strenght)
      for i,test in  enumerate(generate_covering_array.array):
          list_attributes_to_consider = []

          check_for_all_cero = True
          for (test_key, test_value) in test.items():
              if test_value == 1:
                  check_for_all_cero = False
                  list_attributes_to_consider.append(test_key)

          if check_for_all_cero:
              continue  
          lst_of_featutres_to_check.append((list_attributes_to_consider,i))
      
      clf_new = train_model(model)
      results = Parallel(n_jobs=-1)(delayed(run_cv)(clf_new, dataset_X, subset_features, dataset_Y.values.ravel(), i) for subset_features,i in lst_of_featutres_to_check)   
      sorted_results = sorted(results, key=lambda x: x[2])
      
      for score, std, i, subset_features in sorted_results:
        if score >= partial_score:
              partial_score = score
              best_std = std
              best_data_set = subset_features.copy()
      
      if partial_score > global_best_score:
            global_best_score = partial_score
            global_best_features = best_data_set.copy()

      clf_new = None
      best_f1_score = partial_score
      
      if print_logs:
        print(f"best f1 score= {best_f1_score}, iteration:{max_it}, numbers features selected ={ len(best_data_set)},best features selected={', '.join(best_data_set)}" )

      max_it = max_it +1
      max_iteartion_aux = max_iteartion_aux-1
  return global_best_features

def train_model(model_name):  

    m = eval(model_name)
    return m

def run_cv(model, X, subset_features, y, index):

    scores = cross_val_score(model, X[subset_features].values, y, cv=5, scoring='f1_macro',n_jobs=1)
    return (scores.mean(), scores.std(), index, subset_features)

In [25]:
def CAFS(ca,dataset_x,dataset_y, max_iter ,clasifier,print_logs=False):

  max_iteartion = 0
  num_rows = ca.shape[0]
  next_features_data_set_selected = dataset_x.columns.values.copy()

  best_score_global = float('-inf')
  best_features_global = dataset_x.columns.values.copy()

  if len(dataset_x.columns) < ca.shape[1] :
    num_colums = len(dataset_x.columns)
  else:
    num_colums = ca.shape[1]

  while max_iteartion < max_iter:
          
     max_score = 0.0
     mx_data_set = None

     lst_of_subsets_to_run = []
     lst_headers = next_features_data_set_selected.copy()
    
     for i in range(0,num_rows):
        lst_headers_to_select = []
        for j in range(0,num_colums ):
          if ca[i][j] == 1 :
              lst_headers_to_select.append(lst_headers[j])
        if len(lst_headers_to_select) == 0:
            continue
        lst_of_subsets_to_run.append((lst_headers_to_select,i))
     
     results = Parallel(n_jobs=-1)(delayed(run_cv)(clasifier, dataset_x, subset_features, dataset_y.values.ravel(), i) for subset_features,i in lst_of_subsets_to_run)
     sorted_results = sorted(results, key=lambda x: x[2])
     for score, std, i, subset_features in sorted_results:
        if score >= max_score:
              max_score = score
              mx_data_set = subset_features.copy()
              
     if max_score > best_score_global:
          best_score_global = max_score
          best_features_global = mx_data_set.copy()
          
     next_features_data_set_selected = mx_data_set.copy()
     if print_logs:
         print(f"best f1 score= {max_score}, iteration:{max_iteartion}, numbers features selected ={len(mx_data_set)},best features selected={', '.join(mx_data_set)}" )

     num_colums  = len(next_features_data_set_selected)
     max_iteartion = max_iteartion  +1

  return best_features_global, best_score_global

def run_cv(model, X, subset_features, y, index):
    scores = cross_val_score(model, X[subset_features].values, y, cv=5, scoring='f1_macro',n_jobs=1)
    return (scores.mean(), scores.std(), index, subset_features)

In [23]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score
from scipy.stats import wilcoxon

import warnings
warnings.filterwarnings('ignore')

def evaluate_dataset_nested(X, y, dataset_name, outer_splits=5):
    
    print(f"\n{'='*50}")
    print(f"Starting NESTED CV for Dataset: {dataset_name}")
    print(f"{'='*50}")


    # THE OUTER LOOP (The Judge)
    # This splits the data into 'outer_splits' (e.g., 5) folds.
    
    complete_list_of_features = X.columns.values.copy()
    outer_cv = StratifiedKFold(n_splits=outer_splits, shuffle=True, random_state=42)
    
    # Store the scores for every fold
    fold_scores = {'ICAFS': [], 'CAFS': [], 'BBA': [], 'PSO': [], 'CS': []}
    selected_wavelengths = {'ICAFS': [], 'CAFS': [], 'BBA': [], 'PSO': [], 'CS': []}

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y)):
        print(f"\n--- Processing Outer Fold {fold_idx + 1} / {outer_splits} ---")
        
        # 1. Create the Training Data and The Vault for this specific fold
        X_train, X_test = X.iloc[train_idx, :], X.iloc[test_idx, :]
        y_train, y_test = y.iloc[train_idx, :], y.iloc[test_idx, :]
        
        # ---------------------------------------------------------
        # PHASE 1: FEATURE SELECTION (The Scout)
        # ---------------------------------------------------------
        print("  > Running Feature Selection on Training Data...")
        
        
        bba = BBA(num_agents=60, max_iter=20, train_data=X_train, train_label=y_train,save_conv_graph=False,default_mode=True,verbose=False).run()
        pso = PSO(num_agents=60, max_iter=20, train_data=X_train, train_label=y_train,save_conv_graph=False,default_mode=True,verbose=False).run()
        cs = CS(num_agents=60, max_iter=20, train_data=X_train, train_label=y_train,save_conv_graph=False,default_mode=True,verbose=False).run()

        icafs_features = ICAFS(X_train,y_train,2,15,'KNeighborsClassifier()', print_logs=False)
        cafs_features = CAFS(covering_array,X_train,y_train,10,KNeighborsClassifier(),print_logs=False)[0]
        bba_features = [complete_list_of_features[i] for i in range(len(bba.Leader_agent)) if bba.Leader_agent[i] == 1]
        pso_features = [complete_list_of_features[i] for i in range(len(pso.Leader_agent)) if pso.Leader_agent[i] == 1]
        cs_features = [complete_list_of_features[i] for i in range(len(cs.Leader_agent)) if cs.Leader_agent[i] == 1]

        feature_dict = {
            'ICAFS': icafs_features,
            'CAFS': cafs_features,
            'BBA': bba_features,
            'PSO': pso_features,
            'CS': cs_features
        }
        
        # ---------------------------------------------------------
        # PHASE 2 & 3: INNER CV TUNING AND VAULT EVALUATION
        # ---------------------------------------------------------
        print("  > Tuning SVMs and Evaluating on Vault Data...")
        
        svm_param_grid = {
            'C': [0.1, 1, 10, 100],
            'gamma': ['scale', 'auto', 0.01, 0.1],
            'kernel': ['rbf', 'linear']
        }
        
        for algo_name, selected_features in feature_dict.items():
            if len(selected_features) == 0:
                fold_scores[algo_name].append(0.0)
                continue
                
            X_train_masked = X_train[selected_features]
            X_test_masked = X_test[selected_features]
            
            # Phase 2: INNER CV (GridSearchCV automatically splits X_train again internally)
            grid_search = GridSearchCV(SVC(random_state=42), svm_param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
            grid_search.fit(X_train_masked, y_train)
            
            best_svm = grid_search.best_estimator_
            
            # Phase 3: Evaluate on the unseen Test Data (The Vault) for this fold
            y_pred = best_svm.predict(X_test_masked)
            fold_f1 = f1_score(y_test, y_pred, average='macro')
            
            fold_scores[algo_name].append(fold_f1)
            selected_wavelengths[algo_name].append(len(selected_features))

            print(f"    - {algo_name} Fold {fold_idx + 1} F1: {fold_f1:.4f}")
            print(f"  > {algo_name} Final F1-Score: {fold_f1:.4f} (Features: {len(selected_features)})")


    # Calculate the average score across all folds
    final_avg_scores = {algo: np.mean(scores) for algo, scores in fold_scores.items()}
    final_avg_wavelengths = {algo: np.mean(wavelengths_number) for algo, wavelengths_number in selected_wavelengths.items()}
    final_std_features = {algo: np.std(wavelengths_number) for algo, wavelengths_number in selected_wavelengths.items()}
    

    print(f"\n>>> FINAL AVERAGED RESULTS FOR {dataset_name} <<<")
    for algo in final_avg_scores.keys():
        print(f"  {algo}: F1 = {final_avg_scores[algo]:.4f} | Features = {final_avg_wavelengths[algo]:.1f} (± {final_std_features[algo]:.1f})")
    
    return final_avg_scores, final_avg_wavelengths,final_std_features

In [29]:
datasets = [[X_cacao, y_cacao, "Cacao "],[X_meat, y_meat, "Meat"],[X_olive, y_olive, "Olive"],[X_fruit_puree, y_fruit_puree, "Fruit Puree Dataset"],[X_cis,y_cis, "CIS"],[X_algarrobo, y_algarrobo, "Algarrobo"]]
#datasets = [[X_meat, y_meat, "Meat"]]

results = {'Dataset': [], 'ICAFS': [], 'CAFS': [], 'BBA': [], 'PSO': [], 'CS': []}
results_features = {'Dataset': [], 'ICAFS Features': [], 'CAFS Features': [], 'BBA Features': [], 'PSO Features': [], 'CS Features': []}

for X, y, name in datasets:
    scores,num_of_wavelengths,std_final = evaluate_dataset_nested(X, y, name)
    
    results['Dataset'].append(name)
    results['ICAFS'].append(scores['ICAFS'])
    results['CAFS'].append(scores['CAFS'])
    results['BBA'].append(scores['BBA'])
    results['PSO'].append(scores['PSO'])
    results['CS'].append(scores['CS'])

    results_features['Dataset'].append(name)
    results_features['ICAFS Features'].append(f"{num_of_wavelengths['ICAFS']:.1f} (± {std_final['ICAFS']:.1f})")
    results_features['CAFS Features'].append(f"{num_of_wavelengths['CAFS']:.1f} (± {std_final['CAFS']:.1f})")
    results_features['BBA Features'].append(f"{num_of_wavelengths['BBA']:.1f} (± {std_final['BBA']:.1f})")
    results_features['PSO Features'].append(f"{num_of_wavelengths['PSO']:.1f} (± {std_final['PSO']:.1f})")
    results_features['CS Features'].append(f"{num_of_wavelengths['CS']:.1f} (± {std_final['CS']:.1f})")

# Create a DataFrame for easy viewing and exporting to CSV
df_results = pd.DataFrame(results)
df_results_features = pd.DataFrame(results_features)

print("\n==================================================")
print("FINAL RESULTS TABLE (Phase 3 F1-Scores)")
print("==================================================")
print(df_results.to_string(index=False))

print("\n==================================================")
print("FINAL NUMBER OF FEATURES RESULTS TABLE")
print("==================================================")
print(df_results_features.to_string(index=False))


# Export to CSV so you don't lose the data!
df_results.to_csv('final_f1_scores_revision.csv', index=False)
df_results_features.to_csv('final_features_revision.csv', index=False)

# ---------------------------------------------------------
# PHASE 4: STATISTICAL VALIDATION (Reviewer 2 Fix)
# ---------------------------------------------------------
print("\n==================================================")
print("STATISTICAL SIGNIFICANCE (Wilcoxon Signed-Rank Test)")
print("==================================================")

baselines = ['BBA', 'PSO', 'CS']
for baseline in baselines:
    # Compare ICAFS scores against the baseline scores across the 6 datasets
    stat, p_value = wilcoxon(df_results['ICAFS'], df_results[baseline])
    
    print(f"ICAFS vs {baseline}:")
    print(f"  - p-value: {p_value:.4f}")
    if p_value < 0.05:
        print("  - Result: ICAFS is STATISTICALLY SIGNIFICANTLY better.\n")
    else:
        print("  - Result: Difference is not statistically significant (p >= 0.05).\n")
for baseline in baselines:
    # Compare CAFS scores against the baseline scores across the 6 datasets
    stat, p_value = wilcoxon(df_results['CAFS'], df_results[baseline])
    
    print(f"CAFS vs {baseline}:")
    print(f"  - p-value: {p_value:.4f}")
    if p_value < 0.05:
        print("  - Result: CAFS is STATISTICALLY SIGNIFICANTLY better.\n")
    else:
        print("  - Result: Difference is not statistically significant (p >= 0.05).\n")


Starting NESTED CV for Dataset: Cacao 

--- Processing Outer Fold 1 / 5 ---
  > Running Feature Selection on Training Data...
  > Tuning SVMs and Evaluating on Vault Data...
    - ICAFS Fold 1 F1: 1.0000
  > ICAFS Final F1-Score: 1.0000 (Features: 7)
    - CAFS Fold 1 F1: 1.0000
  > CAFS Final F1-Score: 1.0000 (Features: 4)
    - BBA Fold 1 F1: 1.0000
  > BBA Final F1-Score: 1.0000 (Features: 413)
    - PSO Fold 1 F1: 1.0000
  > PSO Final F1-Score: 1.0000 (Features: 604)
    - CS Fold 1 F1: 1.0000
  > CS Final F1-Score: 1.0000 (Features: 421)

--- Processing Outer Fold 2 / 5 ---
  > Running Feature Selection on Training Data...
  > Tuning SVMs and Evaluating on Vault Data...
    - ICAFS Fold 2 F1: 1.0000
  > ICAFS Final F1-Score: 1.0000 (Features: 5)
    - CAFS Fold 2 F1: 0.9909
  > CAFS Final F1-Score: 0.9909 (Features: 11)
    - BBA Fold 2 F1: 1.0000
  > BBA Final F1-Score: 1.0000 (Features: 415)
    - PSO Fold 2 F1: 1.0000
  > PSO Final F1-Score: 1.0000 (Features: 645)
    - CS Fol